# Tải L7 và L8 lên Hugging Face

Chọn kernel `.venv` của dự án rồi chạy **Run All**. Khi `PUBLISH = False`, notebook chỉ đọc và kiểm tra file cục bộ.
Sau khi xem đúng checkpoint, đặt `PUBLISH = True` và chạy lại **Run All** để tải lên repo công khai `solozy/CaroGame`.

Notebook dùng token có quyền ghi trên repo, lấy từ đăng nhập Hugging Face hiện tại hoặc `HF_TOKEN` trong `.env`.
Nếu chưa đăng nhập, dùng `hf auth login` trong terminal. Giữ token ngoài notebook.

L7 dùng đúng trọng số L5 đã đánh giá với PUCT. L8 dùng `best.pt` của vòng 37.

In [ ]:
import hashlib
import json
from pathlib import Path
import re
from tempfile import TemporaryDirectory

import torch
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download, try_to_load_from_cache

from caro.config import load_manifest
from caro.hub import load_dotenv, set_revision
from caro.nn.model import CaroResNet

PUBLISH = False
PIN_LOCAL = True

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "caro").is_dir() and (p / "configs/models.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ dự án CaroGame hoặc thư mục con.")
CONFIG_ROOT = ROOT / "configs"
L8_RUN = ROOT / "runs/train/alphazero/20261124-160615_gpu-native"
REPO = "solozy/CaroGame"
BRANCH = "main"

manifest = load_manifest(CONFIG_ROOT)
if manifest["repo"] != REPO:
    raise ValueError(f"Repo trong configs/models.yaml khác đích dự kiến {REPO}.")

l5 = manifest["models"]["l5"]
l7_path = try_to_load_from_cache(REPO, l5["file"], revision=l5["revision"], repo_type="model")
if not isinstance(l7_path, str):
    raise FileNotFoundError("Chưa có trọng số L5 trong bộ nhớ đệm. Chạy caro-hub pull l5 trong terminal.")

summary = json.loads((L8_RUN / "summary.json").read_text(encoding="utf-8"))
if summary["status"] != "ok" or summary["best_iteration"] != 37:
    raise ValueError("L8 mặc định phải là lượt đã hoàn tất và có best_iteration=37.")

artifacts = {"l7/best.pt": Path(l7_path), "l8/best.pt": L8_RUN / "checkpoints/best.pt"}
digests = {}
for remote, local in artifacts.items():
    state = torch.load(local, weights_only=True, map_location="cpu")
    CaroResNet().load_state_dict(state.get("model_state_dict", state))
    if remote == "l8/best.pt" and state.get("iteration") != summary["best_iteration"]:
        raise ValueError("Checkpoint L8 không khớp vòng tốt nhất ghi trong summary.json.")
    digests[remote] = hashlib.sha256(local.read_bytes()).hexdigest()
    print(f"{remote} · {local.stat().st_size:,} byte · SHA-256 {digests[remote]}")
    print(f"  Nguồn: {local}")

cards = {
    "l7/card.json": {
        "model": "l7", "agent": "L7-PUCT", "checkpoint": "l7/best.pt",
        "weights_source": {"file": l5["file"], "revision": l5["revision"]},
        "sha256": digests["l7/best.pt"],
    },
    "l8/card.json": {
        "model": "l8", "agent": "L8-AlphaZero", "checkpoint": "l8/best.pt",
        "best_iteration": summary["best_iteration"], "run_id": L8_RUN.name,
        "vs_l7_at_200_sims": summary["vs_l7"], "sha256": digests["l8/best.pt"],
    },
}
print(f"Đích: {REPO} · nhánh {BRANCH}")

## Tải lên và xác minh

Hai checkpoint được tải lên cùng một commit trên Hugging Face. Nếu đã có file trên repo, notebook so mã SHA-256
và dừng nếu file khác phiên bản. Vì vậy có thể chạy lại sau khi mất kết nối.
Sau khi tải lại và xác minh đủ hai file, `PIN_LOCAL = True` cập nhật phiên bản của `model:l7` và `model:l8` trong `configs/models.yaml`.

In [ ]:
def publish():
    load_dotenv(ROOT / ".env")
    api = HfApi()
    head = api.model_info(REPO, revision=BRANCH, timeout=20)
    existing = {file.rfilename for file in head.siblings}
    with TemporaryDirectory(prefix="caro-hf-verify-") as cache:
        def verify(remote, revision):
            path = Path(hf_hub_download(REPO, remote, revision=revision, cache_dir=cache))
            if hashlib.sha256(path.read_bytes()).hexdigest() != digests[remote]:
                raise ValueError(f"File trên Hugging Face khác checkpoint đã chọn: {remote}")

        ops = []
        for remote, local in artifacts.items():
            if remote in existing:
                verify(remote, head.sha)
            else:
                ops.append(CommitOperationAdd(remote, str(local)))
        for remote, card in cards.items():
            if remote not in existing:
                payload = json.dumps(card, ensure_ascii=False, indent=2).encode("utf-8")
                ops.append(CommitOperationAdd(remote, payload))

        if ops:
            result = api.create_commit(
                REPO, ops, repo_type="model", revision=BRANCH, parent_commit=head.sha,
                commit_message="Publish Caro L7 PUCT and L8 AlphaZero checkpoints",
            )
            revision = result.oid
            print(f"Đã tải lên: {result.commit_url}", flush=True)
        else:
            revision = head.sha
            print("Cả hai phiên bản đã có trên repo.")

        for remote in artifacts:
            verify(remote, revision)
            print(f"Đã xác minh: https://huggingface.co/{REPO}/blob/{revision}/{remote}")
    return revision


def pin_manifest(revision, root):
    if not re.fullmatch("[0-9a-f]{40}", revision):
        raise ValueError("Revision Hugging Face không phải mã commit hợp lệ.")
    path = root / "models.yaml"
    text = path.read_text(encoding="utf-8")
    models = load_manifest(root)["models"]
    expected = {"l7": "l7/best.pt", "l8": "l8/best.pt"}
    for model, file in expected.items():
        if model in models and models[model]["file"] != file:
            raise ValueError(f"Đường dẫn hiện tại của {model} khác {file}.")
    if "l7" not in models:
        entry = "  l7: {file: l7/best.pt, revision: null}   # PUCT dùng trọng số L5"
        text, count = re.subn(r"(?m)^  l8:", lambda match: entry + "\n" + match[0], text)
        if count != 1:
            raise ValueError("Không tìm thấy vị trí để thêm l7 trong models.yaml.")

    # Dùng bộ cập nhật của dự án, chỉ thay file thật sau khi ghim đủ hai dòng.
    with TemporaryDirectory(dir=root) as folder:
        stage = Path(folder) / path.name
        stage.write_text(text, encoding="utf-8")
        for level in expected:
            set_revision(level, f'"{revision}"', Path(folder))
        stage.replace(path)


if PUBLISH:
    revision = publish()
    if PIN_LOCAL:
        pin_manifest(revision, CONFIG_ROOT)
        print(f"Đã ghim model:l7 và model:l8 tại {revision} trong configs/models.yaml.")
else:
    print("Kiểm tra cục bộ hoàn tất. Đặt PUBLISH = True rồi chạy lại Run All để tải lên.")